### KD tree código

In [8]:
class Node:
    """A node in the 2D kd-tree."""

    def __init__(self, point=None, line_coord=None, axis=None, left=None, right=None):
        self.point = point          # Stored point (for leaf nodes)
        self.line_coord = line_coord  # Median x or y coordinate (for internal nodes)
        self.axis = axis            # Splitting axis: 0 for vertical (x), 1 for horizontal (y)
        self.left = left            # Left/Below child
        self.right = right          # Right/Above child

    def is_leaf(self):
        return self.point is not None


def build_kd_tree(points, depth=0):
    """
    Builds a 2D kd-tree recursively from a list of 2D points (tuples/lists of (x, y)).
    
    :param points: List of points, e.g., [(x1, y1), (x2, y2), ...]
    :param depth: Current depth of the tree (0-indexed)
    :return: Root Node of the kd-tree
    """
    if not points:
        return None

    # Base case: 1 point left -> Leaf node
    if len(points) == 1:
        return Node(point=points[0])

    # Determine axis: Even depth -> x-axis (0), Odd depth -> y-axis (1)
    axis = depth % 2

    # Sort points along the splitting axis to find the median
    points_sorted = sorted(points, key=lambda p: p[axis])
    
    # Median index (lower median for even-length lists to follow convention)
    median_idx = (len(points_sorted) - 1) // 2
    median_coord = points_sorted[median_idx][axis]

    # Split into P1 (left/below or on line) and P2 (right/above line)
    p1 = points_sorted[: median_idx + 1]
    p2 = points_sorted[median_idx + 1 :]

    # Recursively build subtrees
    left_child = build_kd_tree(p1, depth + 1)
    right_child = build_kd_tree(p2, depth + 1)

    # Return internal node storing the splitting line
    return Node(
        line_coord=median_coord,
        axis=axis,
        left=left_child,
        right=right_child
    )
    
    

In [7]:
def search_kd_tree(node, R):
    """
    Algoritmo SEARCHKDTREE(ν, R) simplificado.
    Aplica la lógica del pseudocódigo descartando ramas que no se intersecan con el rango R
    sin requerir objetos geométricos adicionales.
    
    :param node: Raíz o subárbol ν
    :param R: Rango definido como ((xmin, xmax), (ymin, ymax))
    """
    if node is None:
        return []

    # Líneas 1-2: if ν is a leaf -> Report point if it lies in R
    if node.is_leaf():
        x, y = node.point
        (xmin, xmax), (ymin, ymax) = R
        if xmin <= x <= xmax and ymin <= y <= ymax:
            return [node.point]
        return []

    points_found = []
    axis = node.axis
    r_min, r_max = R[axis]

    # Líneas 3-6: Evaluar e inspeccionar hijo izquierdo lc(ν) si el rango cubre/interseca la región izquierda
    if node.left and r_min <= node.line_coord:
        points_found.extend(search_kd_tree(node.left, R))

    # Líneas 7-10: Evaluar e inspeccionar hijo derecho rc(ν) si el rango cubre/interseca la región derecha
    if node.right and r_max >= node.line_coord:
        points_found.extend(search_kd_tree(node.right, R))

    return points_found

In [9]:
# 1. Conjunto de puntos de prueba en 2D (P)
puntos = [(2, 3), (5, 4), (9, 6), (4, 7), (8, 1), (7, 2)]

# 2. Construcción del árbol llamando a BUILDKDTREE(P, depth=0)
raiz = build_kd_tree(puntos, depth=0)

# 3. Definición del rango de búsqueda R: [(xmin, xmax), (ymin, ymax)]
# Queremos encontrar los puntos dentro del rectángulo X entre 4 y 8, Y entre 1 y 5
rango_R = ((4, 8), (1, 5))

# 4. Búsqueda de puntos llamando a SEARCHKDTREE(ν, R)
puntos_encontrados = search_kd_tree(raiz, rango_R)

# 5. Resultados
print(f"Puntos de entrada: {puntos}")
print(f"Rango de consulta R: X en [4, 8], Y en [1, 5]")
print(f"Puntos dentro del rango: {puntos_encontrados}")

Puntos de entrada: [(2, 3), (5, 4), (9, 6), (4, 7), (8, 1), (7, 2)]
Rango de consulta R: X en [4, 8], Y en [1, 5]
Puntos dentro del rango: [(5, 4), (7, 2), (8, 1)]
